## AI Agent
AI agent not only responds to prompt, but decides what actions to take to accomplish a goal, takes those actions, observes the result and repeats it. It basically follows the below loop:

We use `langchain4j-agentic` module and define agent using `@Agent` annotation, like:

In [ ]:
public interface TaxAgent {
    @UserMessage("""
            You are a helpful and friendly U.S. tax advisor AI who speaks like a real-world human expert
            Keep answers concise, practical, and beginner-friendly (under 120 words). Use numbered or bulleted steps where helpful.
            Avoid jargon, long paragraphs, and excessive detail. Always Offer the user a clear next step if possible.
            Always sound calm, approachable, and reassuring. Format output for clarity.
            Format your responses for readability, using lists for advice.
            Answer: {{prompt}}""")
    @Agent("Generates tax related information for U.S. region.")  // Agents description
    String generateTaxReport(@V("prompt") String prompt);
}

Then use `AgenticServices`:

In [ ]:
TaxAgent taxAgent = AgenticServices
    .agentBuilder(TaxAgent.class)
    .chatModel(openaiChatModel)
    .outputKey("taxReport")  // This is where output of this agent is stored. Multiple agents
    .build();                // can be chained together each accessing the output of other
                             // using this key.

## Agentic Scope
Is collection of data shared amongst the agents that make up the agentic system. Langchain4J provides the ability to sequence multiple agents that share the agentic scope.

### Sequential
Multiple agents can be placed in a linear sequence, one using the output of other:

In [ ]:
// ---- Agent 1: condense a raw resume into a short summary ----
interface ResumeSummarizer {
    @SystemMessage("You are an HR assistant that extracts key facts from resumes.")
    @UserMessage("""
        Summarize the candidate's experience, skills, and years in industry
        in no more than 5 bullet points.
        Resume:
        {{resume}}
        """)
    @Agent("Summarizes a raw resume into key bullet points")
    String summarize(@V("resume") String resume);
}
ResumeSummarizer resumeSummarizer = AgenticServices
    .agentBuilder(ResumeSummarizer.class)
    .chatModel(chatModel)
    .outputKey("summary")
    .build();


// ---- Agent 2: compare the summary against a job description ----
interface SkillMatcher {
    @SystemMessage("You are a recruiter evaluating candidate fit.")
    @UserMessage("""
        Compare the candidate summary below against the job requirements.
        List matched skills, missing skills, and an overall fit score from 0-100.

        Candidate summary:
        {{summary}}

        Job requirements:
        {{jobDescription}}
        """)
    @Agent("Matches a candidate summary against a job description")
    String matchSkills(@V("summary") String summary, @V("jobDescription") String jobDescription);
}
SkillMatcher skillMatcher = AgenticServices
    .agentBuilder(SkillMatcher.class)
    .chatModel(chatModel)
    .outputKey("matchReport")
    .build();


// ---- Agent 3: turn the match report into candidate-facing feedback ----
interface FeedbackWriter {
    @SystemMessage("You write constructive, encouraging feedback for job applicants.")
    @UserMessage("""
        Turn the following recruiter match report into a short, kind
        piece of feedback for the candidate. Mention 1-2 strengths and
        1-2 areas to improve. Keep it under 100 words.

        Match report:
        {{matchReport}}
        """)
    @Agent("Writes candidate-facing feedback from a recruiter match report")
    String writeFeedback(@V("matchReport") String matchReport);
}
FeedbackWriter feedbackWriter = AgenticServices
    .agentBuilder(FeedbackWriter.class)
    .chatModel(chatModel)
    .outputKey("feedback")
    .build();


// ---- Typed interface for the whole pipeline ----
interface ScreeningPipeline {
    @Agent
    String screen(@V("resume") String resume, @V("jobDescription") String jobDescription);
}
ScreeningPipeline pipeline = AgenticServices
    .sequenceBuilder(ScreeningPipeline.class)
    .subAgents(resumeSummarizer, skillMatcher, feedbackWriter)
    .outputKey("feedback")         // final result returned to the caller
    .build();

String feedback = pipeline.screen(resume, jobDescription);

## Loop
We use this variant to repeatedly invoke LLM to refine result until we reach certain goal.

In [ ]:
// ---- Agent 1: initial draft of a SQL query from a natural-language request ----
interface QueryWriter {
    @SystemMessage("You are a SQL expert writing queries against a Postgres schema.")
    @UserMessage("""
            Write a SQL query that answers this request:
            {{request}}
            
            Schema:
            {{schema}}
            """)
    @Agent("Writes a first-draft SQL query for a natural language request")
    String writeQuery(@V("request") String request, @V("schema") String schema);
}
QueryWriter queryWriter = AgenticServices
        .agentBuilder(QueryWriter.class)
        .chatModel(openaiChatModel)
        .outputKey("query")
        .build();

// ---- Agent 2: scores the current query for correctness/efficiency ----
interface QueryCritic {
    @SystemMessage("You are a senior DBA reviewing SQL queries for correctness and performance.")
    @UserMessage("""
            Score the following SQL query between 0.0 and 1.0 based on how correctly
            and efficiently it answers the request, given the schema.
            Return only the numeric score and nothing else.
            
            Request: {{request}}
            Schema: {{schema}}
            Query: {{query}}
            """)
    @Agent("Scores a SQL query for correctness and efficiency")
    double scoreQuery(@V("request") String request, @V("schema") String schema, @V("query") String query);
}
QueryCritic queryCritic = AgenticServices
        .agentBuilder(QueryCritic.class)
        .chatModel(openaiChatModel)
        .outputKey("score")
        .build();
 
// ---- Agent 3: rewrites the query to address issues, given the current score ----
interface QueryRewriter {
    @SystemMessage("You are a SQL expert fixing and optimizing queries.")
    @UserMessage("""
            The following SQL query scored {{score}} out of 1.0 for correctness
            and efficiency against this request and schema. Rewrite it to fix
            any issues and improve performance. Return only the corrected SQL
            query and nothing else.
            
            Request: {{request}}
            Schema: {{schema}}
            Query: {{query}}
            """)
    @Agent("Rewrites a SQL query to improve its score")
    String rewriteQuery(@V("request") String request, @V("schema") String schema,
                        @V("query") String query, @V("score") Double score);
}
QueryRewriter queryRewriter = AgenticServices
        .agentBuilder(QueryRewriter.class)
        .chatModel(openaiChatModel)
        .outputKey("query")   // overwrites "query" with the improved version
        .build();

// UnTypedAgent doesn't require an explicit interface
// Looping agent
UntypedAgent refinementLoop = AgenticServices
        .loopBuilder()
        .subAgents(queryCritic, queryRewriter)
        .maxIterations(5)
        .exitCondition(scope -> scope.readState("score", 0.0) >= 0.9)
        // .exitCondition((scope, loopCount) -> scope.readState("score", 0.0) >= 0.9) can be used to get current loop count
        .build();

// Sequential agent
interface SqlAssistant {
    @Agent
    String generateOptimalQuery(@V("request") String request, @V("schema") String schema);
}
SqlAssistant sqlAssistant = AgenticServices
        .sequenceBuilder(SqlAssistant.class)
        .subAgents(queryWriter, refinementLoop)
        .outputKey("query")
        .build();

## Parallel
Can be used when two agents can run parallely:

In [ ]:
// ---- Agent 1: checks for security issues ----
interface SecurityReviewer {
    @SystemMessage("You are an application security engineer.")
    @UserMessage("""
        Review the following code for security vulnerabilities
        (e.g. injection, unsafe deserialization, secrets in code).
        List findings as short bullet points, or say "No issues found".

        Code:
        {{code}}
        """)
    @Agent("Reviews code for security vulnerabilities")
    String reviewSecurity(@V("code") String code);
}
SecurityReviewer securityReviewer = AgenticServices
    .agentBuilder(SecurityReviewer.class)
    .chatModel(chatModel)
    .outputKey("security")
    .build();


// ---- Agent 2: checks for performance issues ----
interface PerformanceReviewer {
    @SystemMessage("You are a performance engineer.")
    @UserMessage("""
        Review the following code for performance problems
        (e.g. needless allocations, N+1 queries, blocking calls on
        hot paths). List findings as short bullet points, or say
        "No issues found".

        Code:
        {{code}}
        """)
    @Agent("Reviews code for performance issues")
    String reviewPerformance(@V("code") String code);
}
PerformanceReviewer performanceReviewer = AgenticServices
    .agentBuilder(PerformanceReviewer.class)
    .chatModel(chatModel)
    .outputKey("performance")
    .build();

interface CodeReviewAgent {
    @Agent
    CodeReviewReport review(@V("code") String code);
}
CodeReviewAgent codeReviewAgent = AgenticServices
    .parallelBuilder(CodeReviewAgent.class)
    .subAgents(securityReviewer, performanceReviewer)
    // Optionally cap concurrency / use your own thread pool instead
    // of the framework's default cached thread pool:
    // .executor(Executors.newFixedThreadPool(3))
    .output(scope -> new StringBuilder()
            .append("Security Review:\n")
            .append(scope.readState("security", "No issues found"))
            .append("Performance Review:\n")
            .append(scope.readState("performance", "No issues found"))
            .toString())
    .build();